# Stage 4: BioLangFusion Phase A on CoV-Vac, E. coli and mRNA stability

Does combining DNA-, RNA- and protein-encoder representations improve prediction over a single encoder on these benchmarks? Phase A establishes the original-encoder comparison before [Phase B changes one encoder at a time](tier2_encoder_swaps.ipynb). It also compares local scores with the published BioLangFusion results, while retaining the implementation differences needed to interpret that comparison.

[Stage 3](../stage3/README.md) prepared the benchmark rows and token embeddings, built the models and examined fungal expression. Here we reuse those embeddings for CoV-Vac degradation, E. coli expression classification and mRNA stability, then return to two fungal questions about RNA resolution and prediction-head width. Ab1, the paper's fifth benchmark, is absent from the supplied release; the [dataset audit](../stage3/01_blf_datasets.ipynb) records that limitation. This work belongs to [Gantt task 17](../../docs/Research_Project_Plan_Gantt.xlsx).

**Execution and resources.** This notebook contains training cells as well as analysis of saved results. The device setting selects an available Apple GPU through MPS, otherwise an NVIDIA GPU through CUDA, otherwise the CPU. Reading saved outputs requires no execution. The runtime section distinguishes historical measurements from estimates; this notebook does not establish a current-machine runtime or peak RAM/VRAM requirement. The [Stage 4 overview](README.md#biolangfusion-phase-a) explains its relationship to the separate appendix comparison.

---

## Setup

Prepare the notebook environment using the root README's [dependency instructions](../../README.md#notebook-dependencies), and select that environment as the notebook kernel. Start from within this repository; the path cell locates its Code directory and anchors the cache and result locations there.

### Imports

In [ ]:
# Environment, Python module paths, elapsed time and filesystem paths.
import os
import sys
import time
from pathlib import Path

# Set before importing torch: unsupported Apple/MPS operations may fall back to CPU.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Numerical arrays (NumPy) and result tables (pandas).
import numpy as np
import pandas as pd

# Tensor computation, model training and accelerator selection.
import torch

# Table display characters and columns, unrelated to sequence or token lengths.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


### Compute device


In [1]:
# Check both accelerator backends; availability depends on this Python environment.
cuda_available = torch.cuda.is_available()
mps_available = torch.backends.mps.is_available()

# Retain the experiment's device preference: Apple/MPS, NVIDIA/CUDA, then CPU.
if mps_available:
    DEVICE = "mps"
elif cuda_available:
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print("PyTorch:", torch.__version__)
print("NVIDIA/CUDA available:", cuda_available)
print("Apple/MPS available:", mps_available)
print("Selected compute device:", DEVICE)
if DEVICE == "cuda":
    print("NVIDIA GPU:", torch.cuda.get_device_name())


torch 2.12.1 | Apple GPU available: True


### Project paths and shared code

Locate the project's Code directory and import the shared helpers for cached inputs, model construction, training and evaluation. Cache and result paths are anchored to this directory. The maintained implementation is in [the fusion helpers](../shared_code/fusion/).

In [ ]:
# Search the kernel's working folder and its ancestors for either layout:
# Code/shared_code from the repository root, or shared_code from inside Code.
working_dir = Path.cwd().resolve()
for parent in (working_dir, *working_dir.parents):
    if (parent / "shared_code").is_dir():
        CODE_DIR = parent
        break
    if (parent / "Code" / "shared_code").is_dir():
        CODE_DIR = parent / "Code"
        break
else:
    # A for-loop's else runs only when no match ends the search with break.
    raise FileNotFoundError(
        "Cannot locate Code/shared_code. Start this notebook from within the repository."
    )

# Let Python find shared_code without changing the kernel's working folder.
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Cached benchmark streams and model construction; training and result evaluation.
from shared_code.fusion import blf, train


### Published reference values

The tables below transcribe [BioLangFusion Tables 1 and 4](../../Papers/core/BioLangFusion.pdf). They are reference values, not results generated by this notebook. Table 1 includes other encoders for context; the local sweep uses Nucleotide Transformer v2 100M, RNA-FM and ESM-2 8M. Table 4 supplies the gated-fusion comparison with and without its entropy term; its shared-projection row is retained as reference material but is not a separate fit in this sweep.

Spearman correlation measures how well predicted and supplied target rankings agree for the regression tasks. E. coli uses accuracy against the stored class IDs. Published E. coli scores are percentages; local result files store proportions, which the comparison code multiplies by 100 for display. Differences between those percentages are percentage points.

In [ ]:
TABLE1 = pd.DataFrame(
    [
        ("Evo",                     "other",    0.653, 0.579, 42.556, 0.403, 0.360),
        ("SpliceBERT",              "other",    0.802, 0.778, 48.455, 0.522, 0.718),
        ("ESM-2 (650M)",            "other",    0.825, 0.734, 46.348, 0.536, 0.679),
        ("ESM-2 (3B)",              "other",    0.772, 0.721, 46.208, 0.537, 0.700),
        ("ESM-2 (8M)",              "single",   0.806, 0.695, 49.017, 0.539, 0.711),
        ("RNA-FM",                  "single",   0.841, 0.767, 52.949, 0.553, 0.743),
        ("Nucleotide Transformer",  "single",   0.780, 0.804, 41.292, 0.530, 0.732),
        ("Concatenation",           "fusion",   0.831, 0.805, 50.280, 0.539, 0.764),
        ("MIL + Entropy",           "fusion",   0.864, 0.824, 52.107, 0.563, 0.760),
        ("Cross Attention",         "fusion",   0.828, 0.812, 53.932, 0.550, 0.765),
    ],
    columns=["encoding", "kind", "cov_vac_degradation", "fungal_expression",
             "ecoli_proteins", "mrna_stability", "ab1"],
).set_index("encoding")

TABLE4 = pd.DataFrame(
    [
        ("MIL without entropy",                     0.854, 0.826, 51.960, 0.556, 0.753),
        ("MIL with shared projection (+ entropy)",  0.859, 0.813, 50.140, 0.563, 0.755),
        ("MIL + entropy",                           0.864, 0.824, 52.107, 0.563, 0.760),
    ],
    columns=["ablation", "cov_vac_degradation", "fungal_expression",
             "ecoli_proteins", "mrna_stability", "ab1"],
).set_index("ablation")


The lookup below connects local variant names to the paper's rows for later comparisons. The additional codon-grid RNA baseline has no published row, so `published` returns `NaN` for it. These reference values do not determine which models are fitted; the candidate settings below do that.

In [2]:
# Our variant names against the paper's rows. The gridded RNA baseline has no published row.
PUBLISHED_ROW = {
    "unimodal:dna": ("Nucleotide Transformer", TABLE1),
    "unimodal:rna": ("RNA-FM", TABLE1),
    "unimodal:prot": ("ESM-2 (8M)", TABLE1),
    "concat": ("Concatenation", TABLE1),
    "mil": ("MIL + Entropy", TABLE1),
    "mil (lam 0)": ("MIL without entropy", TABLE4),
    "cross": ("Cross Attention", TABLE1),
}


def published(variant, key):
    if variant not in PUBLISHED_ROW:
        return float("nan")
    row, table = PUBLISHED_ROW[variant]
    return float(table.loc[row, key])


display(TABLE1)
display(TABLE4)

,kind,cov_vac_degradation,fungal_expression,ecoli_proteins,mrna_stability,ab1
encoding,,,,,,
Evo,other,0.653,0.579,42.556,0.403,0.360
SpliceBERT,other,0.802,0.778,48.455,0.522,0.718
ESM-2 (650M),other,0.825,0.734,46.348,0.536,0.679
ESM-2 (3B),other,0.772,0.721,46.208,0.537,0.700
ESM-2 (8M),single,0.806,0.695,49.017,0.539,0.711
RNA-FM,single,0.841,0.767,52.949,0.553,0.743
Nucleotide Transformer,single,0.780,0.804,41.292,0.530,0.732
Concatenation,fusion,0.831,0.805,50.280,0.539,0.764
MIL + Entropy,fusion,0.864,0.824,52.107,0.563,0.760


,cov_vac_degradation,fungal_expression,ecoli_proteins,mrna_stability,ab1
ablation,,,,,
MIL without entropy,0.854,0.826,51.960,0.556,0.753
MIL with shared projection (+ entropy),0.859,0.813,50.140,0.563,0.755
MIL + entropy,0.864,0.824,52.107,0.563,0.760


### Comparison settings and file locations

Retain the benchmark choices, model dimensions and training limits below. Comments identify each setting's role; the following candidate cell defines which models participate. The training section explains how validation selects configurations for repeated fits.

Read the token caches produced by [Stage 3](../stage3/02_token_embeddings.ipynb), save completed fits under `results/phaseA/<benchmark>/`, and use Stage 3's fungal results as the diagnostic reference. All three paths are anchored to `CODE_DIR`.

In [ ]:
# Benchmarks in the retained execution order.
BENCHMARKS = ("cov_vac_degradation", "ecoli_proteins", "mrna_stability")

# Read Stage 3 inputs and references; save this experiment's results in Stage 4.
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "phaseA"
FUNGAL_STAGE3 = CODE_DIR / "stage3" / "results" / "fungal_expression"

# Trainable model dimensions.
TCONV = "equation"  # DNA upsampling: kernel 2, stride 2, padding 0.
CHANNELS = 100  # Output channels per TextCNN convolution, not encoder embedding width.

# Optimization limits.
BATCH_SIZE = 32  # Maximum sequences per batch, for training and evaluation.
MAX_EPOCHS = 500  # Maximum passes through training data; early stopping can finish sooner.

# Positive gate-entropy penalties tested on validation; zero is a separate ablation.
LAMBDAS = (0.01, 0.5, 1.0)
FUNGAL_LAMBDA = 0.01  # Reuse Stage 3's validation-selected fungal penalty.

# Repeated training fits on the same supplied data splits.
SEEDS = (0, 1, 2)  # CoV-Vac and E. coli.
STABILITY_SEEDS = (0,)  # Retained stability default; additional seeds repeat selected models.


### Models in the initial sweep

The pretrained encoders stay frozen. Their saved token vectors feed trainable prediction models: single-encoder baselines use a convolutional TextCNN head; fusion models first combine streams through concatenation, gated fusion (MIL), or cross-attention. The experiment fits these prediction and fusion components rather than fine-tuning the encoders.

The three native-resolution baselines read each encoder's token stream directly. The fourth, `unimodal-grid:rna`, averages RNA-FM's nucleotide positions in groups of three, matching the RNA resolution supplied to fusion. This local resolution control competes with the other single encoders during validation selection.

`SINGLE_VARIANTS` lists the eligible single-encoder references; `FUSION_VARIANTS` lists the fusion heads being compared. `SWEEP` gives the initial fit order, including each positive MIL penalty and its zero-penalty ablation. Later training seeds repeat only the selected configurations.

In [ ]:
# Native RNA and codon-grid RNA are separate eligible single-encoder baselines.
SINGLE_VARIANTS = ("unimodal:dna", "unimodal:rna", "unimodal:prot", "unimodal-grid:rna")
FUSION_VARIANTS = ("concat", "mil", "cross")

# Exact initial fit order; it differs from the display/comparison order above.
SWEEP = (
    [
        ("unimodal:prot", {}),
        ("unimodal:dna", {}),
        ("unimodal:rna", {}),
        ("unimodal-grid:rna", {}),
        ("concat", {}),
    ]
    + [("mil", {"entropy_lambda": lam}) for lam in LAMBDAS]
    + [("mil", {"entropy_lambda": 0.0}), ("cross", {})]
)


---

## Prepare the benchmark inputs

Each benchmark row pairs a supplied nucleotide sequence with its target and train/validation/test assignment. DNA and RNA encoders receive alphabet-specific forms of that sequence; the protein encoder receives a computational translation. These are three derived representations of one sequence, not three independently measured assays. In particular, translating CoV-Vac fragments is an input-construction assumption, not evidence that each fragment is a biological coding sequence.

CoV-Vac uses supplied degradation targets, mRNA stability uses supplied stability targets, and E. coli uses expression class IDs. Their scores answer different prediction questions. The [dataset descriptions](../../README.md#other-biolangfusion-benchmarks) and [E. coli target guide](../../docs/methods/biological-targets-and-controls.md#protein-expression-in-e-coli) give the source and label limitations.

The next cell opens the Stage 3 token caches alongside the retained benchmark rows. `BenchmarkStreams` checks row counts and expected token lengths; these checks do not independently recover the biological identity or provenance of every row. It reports feature widths and split sizes before training.

In [3]:
# Open row metadata and token caches; extraction belongs to Stage 3.
STREAMS = {key: blf.BenchmarkStreams.open(key, CACHE_ROOT) for key in BENCHMARKS}
for key, streams in STREAMS.items():
    sizes = {s: len(streams.split_rows(s)) for s in ("train", "val", "test")}
    print(f"{key:<20} {streams.task:<14} widths {streams.widths}  splits {sizes}")
print(f"device {DEVICE}; {len(SWEEP)} runs per benchmark at seed 0")


cov_vac_degradation  regression     widths {'dna': 512, 'rna': 640, 'prot': 320}  splits {'train': 1600, 'val': 400, 'test': 400}
ecoli_proteins       classification widths {'dna': 512, 'rna': 640, 'prot': 320}  splits {'train': 3010, 'val': 728, 'test': 712}
mrna_stability       regression     widths {'dna': 512, 'rna': 640, 'prot': 320}  splits {'train': 16742, 'val': 3608, 'test': 3579}
device mps; 10 runs per benchmark at seed 0


**Reading the input summary.** `widths` gives the number of numerical features per token for each encoder, not the number of tokens or sequence length. `splits` reports the retained rows used for fitting, model selection and final evaluation. The selected `device` comes from the settings above; an existing saved device message describes its producing run.

### Interpreting the runtime estimate

The following calculation uses median durations from saved Stage 3 fungal runs and scales them by the total nucleotide count in each benchmark's training split. It is a rough planning proxy. The timing reader matches channel width only; it does not match convolution settings, producing hardware or software environment. Different sequence lengths, model variants and early-stopping epochs can change cost substantially.

For later training seeds, the projection uses DNA-only and MIL-at-0.01 timings as placeholders for the configurations that validation will eventually select. It also uses native RNA timing for the codon-grid RNA control. Missing timings can become `NaN`; the summary's count is the number of available timing estimates, not a count of completed fits. The printed total neither accounts for current saved-result reuse nor estimates RAM/VRAM.

**Historical context.** The saved estimate below is about 11 hours. The [October 19–20 execution record](../../docs/run-records.md#2026-10-20-stage4-blf-phasea-biolangfusion-phase-a-on-cov-vac-e-coli-and-mrna-stability-research-plan-item-1) instead records 26.85 hours on Julian's Mac using MPS, with roughly 19 hours spent on stability. Its additional fungal-time allowance is also historical. These observations explain the estimate's limitations; they are not a forecast for the selected device.

In [ ]:
def fungal_minutes():
    """Median minutes per fungal run for each variant, from Stage 3's stored results."""
    minutes = {}
    for path in FUNGAL_STAGE3.glob("*.json"):
        r = train.Result.load(path)
        # This timing proxy matches width only, not producer hardware or convolution.
        if r.config["channels"] != CHANNELS:
            continue
        name = r.variant if r.variant != "mil" else f"mil {r.config['entropy_lambda']}"
        minutes.setdefault(name, []).append(r.seconds / 60)
    return {k: float(np.median(v)) for k, v in minutes.items()}


def train_nucleotides(key):
    rows = blf.datasets_module.blf_benchmark_rows(key)
    return rows.loc[rows["Split"] == "train", "length"].sum()


The helpers above define the historical timing lookup and training-nucleotide count. The next cell combines them into the projection; its fixed fungal allowance and placeholder repeat costs should be read with the limitations above.

In [4]:
per_variant = fungal_minutes()
fungal_size = train_nucleotides("fungal_expression")
plan = []
for key in BENCHMARKS:
    scale = train_nucleotides(key) / fungal_size
    seeds = SEEDS if key != "mrna_stability" else STABILITY_SEEDS
    for variant, overrides in SWEEP:
        name = (
            f"mil {overrides['entropy_lambda']}"
            if variant == "mil"
            else "unimodal:rna" if variant == "unimodal-grid:rna" else variant
        )
        plan.append(
            {
                "benchmark": key,
                "run": name if variant != "unimodal-grid:rna" else variant,
                "seed": 0,
                "minutes": per_variant.get(name, np.nan) * scale,
            }
        )
    # Placeholder costs; the repeated single encoder and MIL weight are selected later.
    for seed in seeds[1:]:
        for name in ("unimodal:dna", "concat", "mil 0.01", "cross"):
            plan.append(
                {
                    "benchmark": key,
                    "run": name,
                    "seed": seed,
                    "minutes": per_variant.get(name, np.nan) * scale,
                }
            )
plan = pd.DataFrame(plan)
# count excludes missing timing estimates; this table does not track completed work.
summary = plan.groupby("benchmark", sort=False)["minutes"].agg(["count", "sum"])
summary["hours"] = (summary["sum"] / 60).round(1)
display(summary[["count", "hours"]])
print(f"total about {plan['minutes'].sum() / 60:.1f} hours, plus about half an hour on fungal")


,count,hours
benchmark,,
cov_vac_degradation,18,0.2
ecoli_proteins,18,2.7
mrna_stability,10,8.2


total about 11.0 hours, plus about half an hour on fungal


### Three-epoch training check

This cell performs a fresh three-epoch fit of the protein-only model on CoV-Vac every time it runs. It uses the selected device but does not use the sweep's saved-result reuse or checkpoint path. The small table exposes training loss, validation score and elapsed time before committing to the longer sweep.

A lower final training loss is a useful diagnostic, not a requirement that every epoch decrease and not evidence that the full reproduction is correct.

In [5]:
# Fresh optimization without the sweep checkpoint/reuse path.
probe = train.fit(
    blf.build(STREAMS["cov_vac_degradation"], "unimodal:prot", channels=CHANNELS),
    STREAMS["cov_vac_degradation"],
    train.TrainConfig(
        max_epochs=3, batch_size=BATCH_SIZE, channels=CHANNELS, early_stopping_patience=3
    ),
    DEVICE,
    log=None,
)
curve = pd.DataFrame(probe.history)[["epoch", "train_loss", "val_score"]]
display(curve)
print(
    f"train loss falling: {curve['train_loss'].iloc[-1] < curve['train_loss'].iloc[0]} | "
    f"{probe.seconds / 3:.1f} s per epoch"
)


,epoch,train_loss,val_score
0,0,1.179833,0.484937
1,1,1.131704,0.524037
2,2,1.103861,0.549792


train loss falling: True | 1.9 s per epoch


---

## Fit the models in a defined order

The helper cells below define how to read completed results, select models and execute the sweep. Defining them does not train a model; the final cell in this section makes the training calls.

The execution order is retained:

1. Fit the seed-0 candidate grid on CoV-Vac, then E. coli.
2. On each benchmark, repeat the validation-selected single encoder, concatenation, the validation-selected positive-weight MIL model and cross-attention for the remaining training seeds.
3. Run the fungal codon-grid RNA and wider MIL diagnostics.
4. Fit the stability seed-0 grid, then its selected repeated fits if requested by `STABILITY_SEEDS`.

`train.run` reads a compatible completed result instead of fitting it again. An unfinished fit can resume from a compatible saved checkpoint; the default checkpoint interval is ten epochs. Printed epoch progress is not itself a saved checkpoint. The [runbook](RUNBOOK.md) and [run records](../../docs/run-records.md#apply-the-provenance-workflow) describe execution and recovery. Prevent system sleep during an intended long run.

### Configuration and result loading

The configuration helper carries the chosen model geometry and training limits into each fit. Equation 1's DNA transposed convolution uses kernel 2, stride 2 and padding 0. The [Stage 3 comparison](../../docs/run-records.md#2026-10-17-stage3-tconv-rerun-the-papers-two-transposed-convolution-settings-compared) records why it was retained; the [appendix notebook](tier2_blf_phaseA_appendix.ipynb) preserves the separate historical configuration and analysis.

BioLangFusion's Appendix A.4 code listings give different TextCNN widths: List 1 specifies 1,280 output channels per convolution for MIL, while List 2 specifies 100 for cross-attention. This main comparison retains a common width of 100; the fungal diagnostic alone uses 1,280. List 3 supplies the retained batch size and epoch limit. These are references to [the paper's code listings](../../Papers/core/BioLangFusion.pdf), not additional notebook settings.

`load` filters result files by convolution choice, channel width and, when requested, training seed. That makes it useful for this comparison, but it is not a full check of input hashes, source version, environment or every training setting. Keep this limitation in mind when placing additional historical records in the same result directory.

In [ ]:
def config(seed=0, **overrides):
    return train.TrainConfig(
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        channels=CHANNELS,
        tconv=TCONV,
        seed=seed,
        **overrides
    )


def results_dir(key):
    return RESULTS_ROOT / key


def load(key, seed=None, channels=CHANNELS):
    """Finished runs for one benchmark under this notebook's settings."""
    runs = []
    for path in sorted(results_dir(key).glob("*.json")):
        r = train.Result.load(path)
        # Analysis filter only; this is not a full input/source identity check.
        if r.config.get("tconv") != TCONV or r.config["channels"] != channels:
            continue
        if seed is None or r.config["seed"] == seed:
            runs.append(r)
    return runs


### Select the single encoder and entropy weight

The seed-0 validation scores choose two things independently: the strongest of the four single-encoder baselines, and the strongest MIL configuration among positive entropy weights. Here “strongest” means the highest validation metric; it does not refer to test performance.

The entropy weight scales a penalty on the learned gate weights. A positive penalty favors more concentrated weights; zero removes that penalty and serves as the separate ablation. `LAMBDAS` supplies the positive candidates fitted by the initial sweep; `choices` selects among saved positive-weight MIL runs.

`claim_runs` combines the selected single encoder and MIL weight with concatenation and cross-attention. `SEEDS` requests repeated training fits on CoV-Vac and E. coli; `STABILITY_SEEDS` does the same for stability, retaining its one-seed default. These seeds change training randomness, not the supplied data splits. The choices from seed 0 remain fixed for the repeats; the zero-weight MIL ablation stays a seed-0 comparison. More seeds add training work, without an established runtime for another machine.

In [ ]:
def choices(key):
    """The best single encoder and the gated head's lambda, both picked on validation at seed 0."""
    # Fix both choices from seed-0 validation; later training seeds reuse those choices.
    seed0 = load(key, seed=0)
    singles = [r for r in seed0 if r.variant in SINGLE_VARIANTS and r.val_score is not None]
    gated = [
        r
        for r in seed0
        if r.variant == "mil" and r.config["entropy_lambda"] > 0 and r.val_score is not None
    ]
    # max keeps the first loaded candidate on an exact validation-score tie.
    best = max(singles, key=lambda r: r.val_score).variant if singles else None
    lam = max(gated, key=lambda r: r.val_score).config["entropy_lambda"] if gated else None
    return best, lam


def claim_runs(key):
    best, lam = choices(key)
    if best is None or lam is None:
        raise RuntimeError(f"{key}: the seed-0 sweep has not finished")
    return [(best, {}), ("concat", {}), ("mil", {"entropy_lambda": lam}), ("cross", {})]


### Define the training steps

These helpers connect the retained candidate lists to `train.run`. `seed0_sweep` visits the full initial grid; `more_seeds` visits only the selected configurations. `fungal_diagnostics` opens the fungal caches, requests the codon-grid RNA control and the wider MIL fit, then closes those cache handles.

In [ ]:
def seed0_sweep(key):
    print(f"\n=== {key}: seed-0 sweep ===")
    for variant, overrides in SWEEP:
        train.run(STREAMS[key], variant, config(**overrides), DEVICE, results_dir(key))


def more_seeds(key, seeds):
    for seed in seeds[1:]:
        print(f"\n=== {key}: seed {seed} ===")
        for variant, overrides in claim_runs(key):
            train.run(
                STREAMS[key], variant, config(seed=seed, **overrides), DEVICE, results_dir(key)
            )


def fungal_diagnostics():
    print("\n=== fungal_expression: the two open questions from Stage 3 ===")
    fungal = blf.BenchmarkStreams.open("fungal_expression", CACHE_ROOT)
    train.run(fungal, "unimodal-grid:rna", config(), DEVICE, results_dir("fungal_expression"))
    wide = config(entropy_lambda=FUNGAL_LAMBDA)
    # This diagnostic alone uses the wider prediction head.
    wide.channels = 1280
    train.run(fungal, "mil", wide, DEVICE, results_dir("fungal_expression"))
    fungal.close()


### Execute the sweep

This is the result-producing cell. It follows the order described above, reusing compatible completed results or requesting fits through the shared trainer. The elapsed-time message measures this invocation, so a run that reuses results is not a measurement of full retraining cost.

In [6]:
# Calls below can fit models and write results; definitions above do neither.
started = time.perf_counter()
for key in ("cov_vac_degradation", "ecoli_proteins"):
    seed0_sweep(key)
for key in ("cov_vac_degradation", "ecoli_proteins"):
    more_seeds(key, SEEDS)
fungal_diagnostics()
seed0_sweep("mrna_stability")
more_seeds("mrna_stability", STABILITY_SEEDS)
print(f"\nfinished in {(time.perf_counter() - started) / 3600:.2f} hours")



=== cov_vac_degradation: seed-0 sweep ===
  cov_vac_degradation__unimodal-prot__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 1.1823  val 0.4688  best 0.4688 @ 0
    epoch  10  train loss 0.9598  val 0.6369  best 0.6369 @ 10
    epoch  20  train loss 0.8590  val 0.6983  best 0.6983 @ 20
    epoch  30  train loss 0.7709  val 0.7369  best 0.7369 @ 30
    epoch  40  train loss 0.6997  val 0.7561  best 0.7561 @ 40
    epoch  50  train loss 0.6571  val 0.7745  best 0.7745 @ 50
    epoch  60  train loss 0.5921  val 0.7842  best 0.7870 @ 59
    epoch  70  train loss 0.5488  val 0.7969  best 0.7969 @ 70
    epoch  80  train loss 0.4952  val 0.8030  best 0.8040 @ 78
    epoch  90  train loss 0.4648  val 0.8092  best 0.8110 @ 88
    epoch 100  train loss 0.4176  val 0.8120  best 0.8121 @ 99
    epoch 110  train loss 0.4267  val 0.8127  best 0.8129 @ 109
    epoch 120  train loss 0.4259  val 0.8135  best 0.8135 @ 120
    epoch 130  train loss 0.4239  val 0.8132  best 0.81

---

## Compare local scores with the paper

We first ask how the local scores and rankings compare with the published values. This is a descriptive comparison. Close scores alone do not establish identical preprocessing, implementation or training, and a score difference alone does not identify which implementation choice caused it.

The helper below retains the validation-selected positive MIL weight and the zero-weight ablation. It combines available repeated fits only when the variant and entropy weight match. The native DNA, RNA and protein baselines and the additional codon-grid RNA control remain separately visible.

In [ ]:
SCALE = {"ecoli_proteins": 100.0}  # our accuracy is 0 to 1, the paper's 0 to 100


def label(r):
    if r.variant == "mil" and r.config["entropy_lambda"] == 0:
        return "mil (lam 0)"
    return r.variant


def comparison(key):
    scale = SCALE.get(key, 1.0)
    best, lam = choices(key)
    runs = load(key)
    rows = []
    for r in load(key, seed=0):
        if r.variant == "mil" and r.config["entropy_lambda"] not in (0.0, lam):
            continue  # a sweep member validation did not pick
        name = label(r)
        # Average only the same variant and entropy setting across available seeds.
        same = [
            x.test_score
            for x in runs
            if label(x) == name and x.config["entropy_lambda"] == r.config["entropy_lambda"]
        ]
        rows.append(
            {
                "variant": name,
                "lambda": r.config["entropy_lambda"],
                "val": r.val_score * scale if r.val_score is not None else np.nan,
                "test, seed 0": r.test_score * scale,
                "seeds": len(same),
                "test, mean": np.mean(same) * scale,
                "published": published(name, key),
                "epochs": r.epochs,
                "minutes": round(r.seconds / 60, 1),
            }
        )
    order = [*SINGLE_VARIANTS, *FUSION_VARIANTS, "mil (lam 0)"]
    table = pd.DataFrame(rows).set_index("variant")
    table = table.loc[[v for v in order if v in table.index]]
    # Local minus published point estimate, separate from local paired intervals.
    table["difference"] = table["test, mean"] - table["published"]
    return table, best, lam


### Read the comparison tables

`val` and `test, seed 0` refer to the initial fit. `test, mean` averages the matching available fits, and `seeds` gives their count; not every row has the same repetition. `epochs` and `minutes` describe the seed-0 fit rather than that entire repeated set.

`difference` is the local mean test score minus the paper's reported score. It is not the fusion gain against a local single encoder. The following uncertainty section compares local models using saved predictions; it cannot attach an interval to the paper/local difference without the paper's corresponding evidence.

The optional gated-attention line reports the selected seed-0 model's mean weights over its test examples. A test-set mean does not describe every example or establish a stream's causal biological importance.

In [7]:
for key in (*BENCHMARKS, "fungal_expression"):
    if key == "fungal_expression" or not load(key, seed=0):
        continue
    table, best, lam = comparison(key)
    print(f"\n{key}: best single encoder on validation {best}; gated head lambda {lam}")
    display(table.round(3))
    gated = [
        r
        for r in load(key, seed=0)
        if r.variant == "mil" and r.alpha and r.config["entropy_lambda"] == lam
    ]
    if gated:
        # Stored weights are test-set means from this selected seed-0 fit.
        print(
            "  gated attention weights: "
            + "  ".join(f"{k} {v:.2f}" for k, v in gated[0].alpha.items())
        )



cov_vac_degradation: best single encoder on validation unimodal:rna; gated head lambda 0.01


,lambda,val,"test, seed 0",seeds,"test, mean",published,epochs,minutes,difference
variant,,,,,,,,,
unimodal:dna,0.00,0.815,0.779,1,0.779,0.780,55,0.6,-0.001
unimodal:rna,0.00,0.857,0.833,3,0.838,0.841,124,1.6,-0.003
unimodal:prot,0.00,0.814,0.815,1,0.815,0.806,145,1.5,0.009
unimodal-grid:rna,0.00,0.844,0.823,1,0.823,NaN,100,1.3,NaN
concat,0.00,0.823,0.801,3,0.799,0.831,47,0.7,-0.032
mil,0.01,0.800,0.783,3,0.783,0.864,47,0.7,-0.081
cross,0.00,0.854,0.832,3,0.836,0.828,36,1.1,0.008
mil (lam 0),0.00,0.809,0.792,1,0.792,0.854,47,0.7,-0.062


  gated attention weights: dna 0.83  rna 0.01  prot 0.16

ecoli_proteins: best single encoder on validation unimodal:prot; gated head lambda 0.01


,lambda,val,"test, seed 0",seeds,"test, mean",published,epochs,minutes,difference
variant,,,,,,,,,
unimodal:dna,0.00,51.236,47.753,1,47.753,41.292,40,3.0,6.461
unimodal:rna,0.00,53.159,50.702,1,50.702,52.949,31,3.5,-2.247
unimodal:prot,0.00,53.709,50.140,3,51.732,49.017,48,3.8,2.715
unimodal-grid:rna,0.00,51.648,49.298,1,49.298,NaN,26,2.4,NaN
concat,0.00,58.654,60.674,3,58.661,50.280,37,5.8,8.381
mil,0.01,55.082,54.354,3,54.541,52.107,31,4.9,2.434
cross,0.00,56.044,53.792,3,53.511,53.932,41,31.4,-0.421
mil (lam 0),0.00,55.495,55.758,1,55.758,51.960,31,4.7,3.798


  gated attention weights: dna 0.11  rna 0.00  prot 0.89

mrna_stability: best single encoder on validation unimodal:rna; gated head lambda 0.01


,lambda,val,"test, seed 0",seeds,"test, mean",published,epochs,minutes,difference
variant,,,,,,,,,
unimodal:dna,0.00,0.533,0.539,1,0.539,0.530,36,45.1,0.009
unimodal:rna,0.00,0.555,0.552,1,0.552,0.553,91,140.5,-0.001
unimodal:prot,0.00,0.536,0.531,1,0.531,0.539,95,120.7,-0.008
unimodal-grid:rna,0.00,0.550,0.547,1,0.547,NaN,113,150.5,NaN
concat,0.00,0.551,0.558,1,0.558,0.539,51,83.0,0.019
mil,0.01,0.547,0.551,1,0.551,0.563,62,109.7,-0.012
cross,0.00,0.548,0.553,1,0.553,0.550,33,158.6,0.003
mil (lam 0),0.00,0.553,0.558,1,0.558,0.556,62,106.7,0.002


  gated attention weights: dna 0.28  rna 0.00  prot 0.71


**Reading the saved comparison.** In the October 19–20 outputs, CoV-Vac MIL is below its published score, while E. coli concatenation is above its published score. Several single-encoder and cross-attention scores are close to the references. The next table separates these numerical patterns into explicit checks before we examine local uncertainty.

### Point-estimate checks

The `ok` and `NO` labels summarize the stated conditions. The 0.05 tolerance means 0.05 correlation units or five accuracy percentage points; it is a descriptive threshold, not a significance test. Rankings and the entropy-ablation direction use available mean scores, sometimes with unequal seed counts.

In [8]:
# Descriptive checks on mean scores where repeated fits exist, seed 0 otherwise.
# The tolerance and rankings below are not statistical significance tests.
verdict = []
for key in BENCHMARKS:
    if not load(key, seed=0):
        continue
    table, best, _ = comparison(key)
    ours = table["test, mean"].to_dict()
    # 0.05 correlation units, or five percentage points for accuracy.
    tolerance = 0.05 * SCALE.get(key, 1.0)
    have = [v for v in table.index if v in PUBLISHED_ROW]
    gaps = (table.loc[have, "test, mean"] - table.loc[have, "published"]).abs()

    def claim(text, known, passed, detail):
        verdict.append(
            {
                "benchmark": key,
                "claim": text,
                "result": "not yet" if not known else ("ok" if passed else "NO"),
                "detail": detail,
            }
        )

    claim(
        "every variant within 0.05 of its published value",
        bool(have),
        bool((gaps <= tolerance).all()),
        f"largest gap {gaps.max():.3f} ({gaps.idxmax()})",
    )
    plain = [v for v in ("unimodal:dna", "unimodal:rna", "unimodal:prot") if v in ours]
    paper_order = sorted(plain, key=lambda v: -published(v, key))
    our_order = sorted(plain, key=lambda v: -ours[v])
    claim(
        "the three single encoders in the published order",
        len(plain) == 3,
        our_order == paper_order,
        "published " + " > ".join(paper_order) + "; ours " + " > ".join(our_order),
    )
    # A positive point difference can still have a paired interval containing zero.
    heads_run = [h for h in FUSION_VARIANTS if h in ours]
    claim(
        "every fusion head beats the best single encoder (point estimates)",
        len(heads_run) == 3 and best in ours,
        all(ours[h] > ours.get(best, np.inf) for h in heads_run),
        f"best single {best} {ours.get(best, np.nan):.3f}; "
        + ", ".join(f"{h} {ours[h]:.3f}" for h in heads_run),
    )
    paper_best = max(FUSION_VARIANTS, key=lambda h: published(h, key))
    claim(
        f"the paper's best head is ours too ({paper_best})",
        len(heads_run) == 3,
        max(heads_run, key=lambda h: ours[h]) == paper_best,
        ", ".join(f"{h} {ours[h]:.3f}" for h in heads_run),
    )
    if "mil" in ours and "mil (lam 0)" in ours:
        mine = ours["mil"] - ours["mil (lam 0)"]
        theirs = published("mil", key) - published("mil (lam 0)", key)
        claim(
            "entropy moves the gated head the way Table 4 does",
            True,
            (mine > 0) == (theirs > 0),
            f"ours {mine:+.3f}, paper {theirs:+.3f}",
        )
pd.DataFrame(verdict).set_index(["benchmark", "claim"])


result                                             detail
benchmark           claim                                                                                                       
cov_vac_degradation every variant within 0.05 of its published value       NO                            largest gap 0.081 (mil)
                    the three single encoders in the published order       ok  published unimodal:rna > unimodal:prot > unimo...
                    every fusion head beats the best single encoder...     NO  best single unimodal:rna 0.838; concat 0.799, ...
                    the paper's best head is ours too (mil)                NO               concat 0.799, mil 0.783, cross 0.836
                    entropy moves the gated head the way Table 4 does      NO                          ours -0.009, paper +0.010
ecoli_proteins      every variant within 0.05 of its published value       NO                         largest gap 8.381 (concat)
                    the three single encoders in the published order       NO  published unimodal:rna > unimodal:prot > unimo...
                    every fusion head beats the best single encoder...     ok  best single unimodal:prot 51.732; concat 58.66...
                    the paper's best head is ours too (cross)              NO            concat 58.661, mil 54.541, cross 53.511
                    entropy moves the gated head the way Table 4 does      NO                          ours -1.217, paper +0.147
mrna_stability      every variant within 0.05 of its published value       ok                         largest gap 0.019 (concat)
                    the three single encoders in the published order       NO  published unimodal:rna > unimodal:prot > unimo...
                    every fusion head beats the best single encoder...     NO  best single unimodal:rna 0.552; concat 0.558, ...
                    the paper's best head is ours too (mil)                NO               concat 0.558, mil 0.551, cross 0.553
                    entropy moves the gated head the way Table 4 does      NO                          ours -0.007, paper +0.007

---

## What the test splits can resolve

The previous tables compare point estimates. Now we ask whether the local differences retain their direction when the evaluated test examples are resampled. The inputs are the saved per-row predictions and their corresponding test targets; no model is retrained in this section.

For each of 4,000 bootstrap draws, `train.seed_averaged_bootstrap` resamples identical-sequence groups with replacement. All models use the same selected rows, preserving a paired comparison. Each fitted training seed is scored separately on that resample, then those scores are averaged for its model. Predictions are not averaged into an ensemble, and training seeds are not resampled.

For two compared models, the reported point difference is:

$$
\Delta = \overline{S}_A - \overline{S}_B.
$$

**In this notation:**

- $A$ and $B$ identify the first and second models named in a comparison row, such as concatenation and the validation-selected single encoder.
- $\overline{S}_A$ and $\overline{S}_B$ are each model's test scores averaged over its available fitted training seeds.
- $\Delta$ is their score difference: correlation units for regression, or percentage points after the E. coli display scaling.

Positive differences favor the first model. `ci_low` and `ci_high` are the 2.5th and 97.5th percentiles of the resampled differences. `resolved` means that this interval excludes zero. An interval containing zero leaves the direction unresolved; it does not prove equality. The intervals are individual comparisons, without a correction for examining several comparisons.

`first_wins` is the fraction of resampled differences above zero, not a p-value or a probability that a scientific hypothesis is true. The intervals describe test-sample variation conditional on the available fitted models and selection; they do not include uncertainty from new training sets, unobserved seeds or the paper's reported scores.

### Match predictions to test rows

The helpers collect saved predictions and identify test sequences absent from training. The latter provides a second stability evaluation using the same fitted models. The uncertainty calculation checks one seed-0 reference result's test targets against the current split's row order. It also requires prediction arrays of the expected length, but does not independently verify every result's target ordering or input identity.

In [ ]:
def predictions_by_variant(key, variants, lam):
    """Per-seed test predictions for each named variant, keyed by its label."""
    out = {}
    for r in load(key):
        name = label(r)
        if name == "mil" and r.config["entropy_lambda"] != lam:
            continue
        if name in variants and r.test_prediction is not None:
            out.setdefault(name, []).append(np.asarray(r.test_prediction))
    return out


def unseen_test_rows(key):
    """Positions in the test split whose sequence never appears in the training split."""
    rows = STREAMS[key].rows
    train_sequences = set(rows.loc[rows["Split"] == "train", "sequence"])
    test_sequences = rows["sequence"].to_numpy()[STREAMS[key].split_rows("test")]
    return np.flatnonzero([s not in train_sequences for s in test_sequences])


### Calculate paired score differences

`uncertainty` uses the validation choices already fixed at seed 0. It compares each fusion head with the selected single encoder and also compares the fusion heads with one another. The `rows` argument can restrict scoring to the stability subset without changing fitted models or their predictions.

In [ ]:
def uncertainty(key, rows=None):
    best, lam = choices(key)
    variants = (best, *FUSION_VARIANTS)
    preds = predictions_by_variant(key, variants, lam)
    if any(v not in preds for v in variants):
        return None
    streams = STREAMS[key]
    test = streams.split_rows("test")
    truth = streams.labels[test]
    # Check one seed-0 reference target array against the current test-row order.
    reference = np.asarray(load(key, seed=0)[0].test_truth)
    if not np.allclose(reference, truth):
        raise ValueError(f"{key}: stored test labels are not in the split's row order")
    comparisons = [(f"{h} minus best single ({best})", h, best) for h in FUSION_VARIANTS]
    comparisons += [
        ("cross minus mil", "cross", "mil"),
        ("cross minus concat", "cross", "concat"),
        ("mil minus concat", "mil", "concat"),
    ]
    # Resample sequence groups jointly; average scores, not predictions.
    out = pd.DataFrame(
        train.seed_averaged_bootstrap(
            streams.task,
            truth,
            preds,
            comparisons,
            rows=rows,
            groups=streams.rows["sequence"].to_numpy()[test],
            draws=4000,
            seed=0,
        )
    )
    scale = SCALE.get(key, 1.0)
    # Accuracy differences use percentage points; correlations keep their native scale.
    for column in ("difference", "ci_low", "ci_high"):
        out[column] = out[column] * scale
    out["resolved"] = np.where((out["ci_low"] > 0) | (out["ci_high"] < 0), "yes", "no")
    # Fit counts for first/second model, not independent test samples.
    out["seeds"] = [f"{len(preds[a])}/{len(preds[b])}" for _, a, b in comparisons]
    return out.set_index("comparison")


### Read the uncertainty tables

Each table keeps `difference` and its interval together. `rows` is the number of evaluated test observations; `groups` is the number of distinct sequence groups. `seeds` reports the available fit counts for the first and second models. A smaller interval is not evidence about training variation that the saved fits do not represent.

In [9]:
for key in BENCHMARKS:
    table = uncertainty(key) if load(key, seed=0) else None
    if table is None:
        print(f"{key}: not run yet")
        continue
    print(f"\n{key}, full published test split")
    display(table.round(3))
    if key == "mrna_stability":
        unseen = unseen_test_rows(key)
        print(f"\n{key}, the {len(unseen)} test rows whose sequence is not in training")
        display(uncertainty(key, rows=unseen).round(3))



cov_vac_degradation, full published test split


,difference,ci_low,ci_high,first_wins,rows,groups,resolved,seeds
comparison,,,,,,,,
concat minus best single (unimodal:rna),-0.038,-0.066,-0.012,0.002,400,400,yes,3/3
mil minus best single (unimodal:rna),-0.055,-0.093,-0.019,0.001,400,400,yes,3/3
cross minus best single (unimodal:rna),-0.002,-0.026,0.020,0.410,400,400,no,3/3
cross minus mil,0.052,0.028,0.078,1.000,400,400,yes,3/3
cross minus concat,0.036,0.017,0.057,0.999,400,400,yes,3/3
mil minus concat,-0.016,-0.039,0.006,0.072,400,400,no,3/3



ecoli_proteins, full published test split


,difference,ci_low,ci_high,first_wins,rows,groups,resolved,seeds
comparison,,,,,,,,
concat minus best single (unimodal:prot),6.929,4.167,9.597,1.000,712,712,yes,3/3
mil minus best single (unimodal:prot),2.809,0.936,4.822,0.999,712,712,yes,3/3
cross minus best single (unimodal:prot),1.779,-0.890,4.494,0.897,712,712,no,3/3
cross minus mil,-1.030,-3.886,1.826,0.235,712,712,no,3/3
cross minus concat,-5.150,-7.584,-2.715,0.000,712,712,yes,3/3
mil minus concat,-4.120,-6.835,-1.404,0.001,712,712,yes,3/3



mrna_stability, full published test split


,difference,ci_low,ci_high,first_wins,rows,groups,resolved,seeds
comparison,,,,,,,,
concat minus best single (unimodal:rna),0.007,-0.006,0.019,0.851,3579,3021,no,1/1
mil minus best single (unimodal:rna),-0.001,-0.014,0.012,0.450,3579,3021,no,1/1
cross minus best single (unimodal:rna),0.001,-0.012,0.015,0.564,3579,3021,no,1/1
cross minus mil,0.002,-0.009,0.014,0.636,3579,3021,no,1/1
cross minus concat,-0.006,-0.017,0.005,0.166,3579,3021,no,1/1
mil minus concat,-0.007,-0.017,0.002,0.071,3579,3021,no,1/1



mrna_stability, the 1135 test rows whose sequence is not in training


,difference,ci_low,ci_high,first_wins,rows,groups,resolved,seeds
comparison,,,,,,,,
concat minus best single (unimodal:rna),0.040,-0.003,0.083,0.966,1135,1083,no,1/1
mil minus best single (unimodal:rna),0.034,-0.012,0.079,0.920,1135,1083,no,1/1
cross minus best single (unimodal:rna),0.027,-0.019,0.073,0.866,1135,1083,no,1/1
cross minus mil,-0.006,-0.043,0.030,0.359,1135,1083,no,1/1
cross minus concat,-0.013,-0.050,0.024,0.241,1135,1083,no,1/1
mil minus concat,-0.006,-0.039,0.027,0.360,1135,1083,no,1/1


**Reading the saved intervals.** These observations describe the saved October 19–20 comparison, not a new analysis:

| Benchmark and comparison | Saved difference and 95% interval | Reading |
| --- | --- | --- |
| CoV-Vac: concatenation minus RNA-FM | −0.038 [−0.066, −0.012] | Favors the selected RNA-only model |
| CoV-Vac: MIL minus RNA-FM | −0.055 [−0.093, −0.019] | Favors the selected RNA-only model |
| E. coli: concatenation minus ESM-2 | +6.9 [+4.2, +9.6] percentage points | Favors concatenation |
| E. coli: MIL minus ESM-2 | +2.8 [+0.9, +4.8] percentage points | Favors MIL |

Cross-attention's interval against the selected single encoder contains zero on CoV-Vac and E. coli. On stability, every fusion-versus-single interval contains zero on both evaluated subsets. The unseen-sequence subset has positive point differences, including concatenation at +0.040 [−0.003, +0.083], but the direction remains unresolved by that interval.

### Stability: full test split versus sequences absent from training

The [saved split audit](../stage3/01_blf_datasets.ipynb) found training-sequence overlap in 2,444 of the 3,579 stability test rows. The remaining 1,135 rows form the training-sequence-unseen subset used above and below. These counts identify the evaluated dataset snapshot rather than current execution progress.

The next table scores each seed-0 model on both sets. Its `drop` column is actually the signed subset score minus the full-test score, so a reduction is negative. The subset changes both overlap status and the mixture of examples. It cannot by itself isolate how much of the full-test score was caused by memorization. Validation also contains training-sequence overlap, so early stopping has not been made sequence-disjoint.

In [10]:
# Score the same seed-0 predictions on two test sets; no model is retrained.
# The subset changes sample composition too, so this does not isolate memorization.
if load("mrna_stability", seed=0):
    streams = STREAMS["mrna_stability"]
    unseen = unseen_test_rows("mrna_stability")
    rows = []
    for r in load("mrna_stability", seed=0):
        truth, pred = np.asarray(r.test_truth), np.asarray(r.test_prediction)
        name = f"mil (lam {r.config['entropy_lambda']})" if r.variant == "mil" else label(r)
        rows.append(
            {
                "variant": name,
                "lambda": r.config["entropy_lambda"],
                "published": (
                    published(label(r), "mrna_stability")
                    if r.variant != "mil"
                    or r.config["entropy_lambda"] in (0.0, choices("mrna_stability")[1])
                    else np.nan
                ),
                "full test": r.test_score,
                "unseen sequences": train.score_rows(streams.task, truth[unseen], pred[unseen]),
            }
        )
    leak = pd.DataFrame(rows).set_index("variant")
    # Signed difference: a lower score on unseen sequences appears as a negative value.
    leak["drop"] = leak["unseen sequences"] - leak["full test"]
    display(leak.round(3))
else:
    print("mRNA stability not run yet")


,lambda,published,full test,unseen sequences,drop
variant,,,,,
concat,0.00,0.539,0.558,0.359,-0.199
cross,0.00,0.550,0.553,0.347,-0.206
mil (lam 0.01),0.01,0.563,0.551,0.353,-0.198
mil (lam 0.0),0.00,0.556,0.558,0.359,-0.199
mil (lam 0.5),0.50,NaN,0.545,0.327,-0.217
mil (lam 1.0),1.00,NaN,0.549,0.336,-0.214
unimodal:dna,0.00,0.530,0.539,0.266,-0.274
unimodal-grid:rna,0.00,NaN,0.547,0.340,-0.208
unimodal:prot,0.00,0.539,0.531,0.314,-0.217


**Reading the saved stability diagnostic.** RNA-FM decreases from 0.552 on the full test split to 0.319 on the subset; concatenation decreases from 0.558 to 0.359. The other saved variants also score lower on that subset. This limits what the published-split score says about prediction on sequences absent from training. It does not turn the original fit into a newly trained, sequence-disjoint experiment.

---

## Return to the two fungal questions

The earlier training section produced two fungal diagnostics. We now compare their saved scores with the Stage 3 references and the paper; the comparison cell itself performs no fitting.

**RNA resolution.** The original plain RNA-FM baseline scored 0.722 in the saved Stage 3 run, versus the paper's 0.767. The codon-grid baseline asks whether averaging RNA positions in threes changes local performance when the predictor reads the same resolution as the fusion models. A higher score would establish a useful local control, not reveal the paper's unreported baseline implementation.

**Prediction-head width.** The wider MIL diagnostic uses 1,280 channels from the paper's Appendix A.4 code listing List 1, while the main comparisons retain the 100-channel width from code listing List 2. This table compares one wider fit with the available 100-channel fits and the paper's 0.824. Different repetition counts limit how directly a difference can be attributed to width.

In [11]:
# Row labels distinguish Stage 4 diagnostics from Stage 3 references.
fungal_runs = {
    f"{label(r)} @ {r.config['channels']} ch, seed {r.config['seed']}": r
    for r in (train.Result.load(p) for p in sorted(results_dir("fungal_expression").glob("*.json")))
}
stage3 = [train.Result.load(p) for p in sorted(FUNGAL_STAGE3.glob("*.json"))]
reference = {
    "RNA-FM, plain baseline (Stage 3, seed 0)": [
        r.test_score for r in stage3 if r.variant == "unimodal:rna" and r.config["seed"] == 0
    ],
    "gated head, 100 channels (Stage 3, seeds 0 to 2)": [
        r.test_score
        for r in stage3
        if r.variant == "mil"
        and r.config.get("tconv") == "equation"
        and r.config["entropy_lambda"] == FUNGAL_LAMBDA
        and r.config["channels"] == CHANNELS
    ],
}
# References average their available fits; diagnostic rows represent individual fits.
rows = [{"model": k, "test": np.mean(v), "runs": len(v)} for k, v in reference.items() if v]
rows += [{"model": k, "test": r.test_score, "runs": 1} for k, r in fungal_runs.items()]
fungal_table = pd.DataFrame(rows).set_index("model")
display(fungal_table.round(3))
print(
    f"published: RNA-FM {published('unimodal:rna', 'fungal_expression')}, "
    f"gated head {published('mil', 'fungal_expression')}"
)


,test,runs
model,,
"RNA-FM, plain baseline (Stage 3, seed 0)",0.722,1
"gated head, 100 channels (Stage 3, seeds 0 to 2)",0.801,3
"mil @ 1280 ch, seed 0",0.812,1
"unimodal-grid:rna @ 100 ch, seed 0",0.794,1


published: RNA-FM 0.767, gated head 0.824


**Reading the saved fungal diagnostics.** The codon-grid RNA baseline scores 0.794, compared with 0.722 for the plain baseline and 0.767 in the paper. That is evidence that resolution matters in this local comparison; it does not identify the paper's baseline resolution.

The 1,280-channel MIL fit scores 0.812, compared with the three-fit 100-channel mean of 0.801 and the published 0.824. This motivates examining width, but one wider fit versus a three-seed mean does not isolate width's effect or establish that width explains a particular fraction of the gap.

---

## What this comparison establishes

The saved results show that fusion's usefulness depends on the benchmark and fusion head. E. coli supports positive paired differences for concatenation and gated fusion against the selected protein-only baseline. CoV-Vac instead favors RNA-FM over those two heads. Cross-attention and the stability comparisons leave some differences unresolved. These are results for the declared benchmark splits and available training seeds, not a general ranking of multimodal methods.

The original [October 20 interpretation](../../docs/run-records.md#2026-10-20-stage4-blf-phasea-biolangfusion-phase-a-on-cov-vac-e-coli-and-mrna-stability-research-plan-item-1) treated score agreement as evidence that the build matched the paper, and proposed gating, convolution, RNA resolution and width as explanations for the remaining gaps. The numerical observations are retained here, with narrower conclusions: similar scores do not prove implementation identity, and a diagnostic comparison does not establish a unique cause.

The saved mean gating weights are strongly concentrated on particular streams, with very little RNA weight in these examples. Those averages describe the fitted gate; they do not show every sequence's weights, establish biological importance or explain the score gap causally. Likewise, E. coli concatenation's score above the paper motivates investigating implementation differences, but this notebook does not isolate convolution as the cause.

The stability subset exposes much lower performance on sequences absent from training, while the fungal controls show sensitivity to representation resolution and model width. These findings motivate specific checks rather than settling the mechanisms. Cross-study claims about GTEx or IsoFormer belong with those separate experiments in the [Stage 4 overview](README.md), not in this notebook's evidence alone.

## Choices and limitations to retain with the results

The [Stage 3 dataset audit](../stage3/01_blf_datasets.ipynb) and [input/target guide](../../docs/methods/biological-targets-and-controls.md#splits-preprocessing-and-the-claim-being-tested) explain inherited choices: keeping supplied splits, dropping sequences longer than 999 nucleotides, translating complete triplets from the first base with stops represented by `X`, and the limits of matching published preprocessing. Equal retained counts do not establish identical source rows or verified biological reading frames.

| Choice | What the comparison establishes | Remaining limitation |
| --- | --- | --- |
| Equation 1 convolution and a 100-channel TextCNN | A declared local model configuration, with one wider fungal diagnostic | The paper's conflicting implementation descriptions are not resolved by matching a score |
| Full stability test split plus its training-sequence-unseen subset | Two evaluations of the same fitted models | Subset composition changes, and validation still contains training-sequence overlap; this is not a new sequence-disjoint training experiment |
| One stability training seed by default | Test-sample uncertainty conditional on that fit | It does not estimate stability's variation across training seeds; fungal seed spread cannot establish that variation |
| Validation selection among four single-encoder baselines | Fusion is compared with the selected native or codon-grid baseline | The paper has no separate gridded-RNA reference row; the local baseline selection need not match its procedure |
| Selection at seed 0, with only selected configurations repeated | Repetition focuses on the declared local comparisons | The later seeds do not repeat the entire selection process |
| Identical sequences resampled together | Exact repeated sequences do not count as independent bootstrap groups | Sequence grouping does not account for all gene-family, species or assay dependence |
| Saved results read using limited configuration filters | The declared tables can be assembled from available local records | The reader does not independently establish complete input and producer provenance |

The stored class IDs for E. coli and supplied scalar regression targets are evaluated as provided. Their biological units and transformations should be taken from the [dataset descriptions](../../README.md#data) and [target guide](../../docs/methods/biological-targets-and-controls.md#fungal-and-e-coli-benchmark-targets), rather than inferred from the displayed scores.

---

## How Phase B uses this comparison

[The encoder-swap notebook](tier2_encoder_swaps.ipynb) asks what changes when one of the original pretrained encoders is replaced. Phase A supplies the original-encoder scores, selected entropy settings and comparison context. The [Stage 4 overview](README.md#phase-b-changing-one-encoder-at-a-time) describes which settings Phase B inherits and how its results are interpreted.

Keep the selected single-encoder reference and its RNA resolution explicit when comparing fusion gains. The fungal control demonstrates that resolution can matter locally; it does not establish which baseline the paper used. Missing benchmarks, sequence overlap, wider-head comparisons and additional training seeds remain distinct questions, not work performed by an encoder swap.